# 📖 《Master of the Senate: The Years of Lyndon Johnson, Vol.3》出版级自动化翻译管线
### 🚀 模型架构：Google Gemma 3 27B IT（专为 40GB A100 GPU 量化调优）

**配置特性**：
1. **官方正统 Google Gemma 3 27B**：采用 `google/gemma-3-27b-it` 最新一代多模态与多语言旗舰；
2. **40GB A100 显存专项优化**：启用 `bitsandbytes` 4-bit 深度量化与内存边界保护，显存占用压至 ~20GB，彻底杜绝 54GB+ 原生权重引发的 OOM 显存溢出；
3. **每次执行彻底重置缓存（Clean Slate 机制）**：运行即自动清除历史 `cache/` 与 `raw/` 目录，从原书重新解包，100% 完整全新重译全书，绝不残留旧数据；
4. **出版级 GB/T 15834 排版标准**：正文统一注入 2em 首行缩进，豁免各级标题、引言、插图与表格，保留全部 DOM 原生链接与样式。


In [ ]:
# ==========================================
# 1. 挂载 Google Drive & 安装高性能运行环境
# ==========================================
import os
from google.colab import drive

# 智能防重挂载
if not os.path.exists('/content/drive/MyDrive'):
    print('👉 请在弹出的窗口中点击【连接到 Google 云端硬盘】！')
    try:
        drive.mount('/content/drive', force_remount=True)
    except Exception as e:
        print(f'挂载提示: {e}')
else:
    print('✓ Google Drive 已经挂载就绪！')

# 安装推理库、量化库与 Hugging Face 工具（bitsandbytes 支持 40G 显存运行 27B）
!pip install -q vllm bitsandbytes accelerate huggingface_hub

# 克隆/更新底层核心工程工具库
if not os.path.exists('/content/epub_translate'):
    !git clone https://github.com/bohemianer/epub_translate.git /content/epub_translate
else:
    %cd /content/epub_translate
    !git pull

%cd /content/epub_translate
print('✓ 运行环境与核心脚本准备就绪！')


In [ ]:
# ==========================================
# 2. 载入 Google Gemma 3 27B 旗舰模型（40G A100 优化版）
# ==========================================
import os
import torch
from huggingface_hub import login

# 关键修复：关闭 vLLM 冲突的日志劫持
os.environ['VLLM_CONFIGURE_LOGGING'] = '0'

# Gemma 3 为门控模型，优先读取 Colab Secrets 中的 HF_TOKEN
hf_token = os.environ.get('HF_TOKEN', '')
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN') or hf_token
except Exception:
    pass

if hf_token:
    login(token=hf_token)
    print('✓ Hugging Face 授权登录成功！')
else:
    print('👉 提示：Gemma 3 需要在 huggingface.co/google/gemma-3-27b-it 接受许可协议。')
    print('   建议在 Colab 侧边栏【🔑 密钥 (Secrets)】中添加 HF_TOKEN，或在此处直接填入：')
    manual_token = input('   请输入您的 Hugging Face Token（若已全局授权直接回车）：').strip()
    if manual_token:
        login(token=manual_token)

from vllm import LLM, SamplingParams

MODEL_NAME = 'google/gemma-3-27b-it'

print(f'正在将 Google Gemma 3 27B ({MODEL_NAME}) 载入 40G A100 GPU 显存...')
print('已启用 bitsandbytes 4-bit 量化，显存占用约 20GB，留足 20GB 缓冲空间防 OOM。')

llm = LLM(
    model=MODEL_NAME,
    quantization='bitsandbytes',
    load_format='bitsandbytes',
    tensor_parallel_size=1,
    gpu_memory_utilization=0.85,
    max_model_len=4096,
    trust_remote_code=True,
    enforce_eager=True
)

# Gemma 3 专用采样参数
sampling_params = SamplingParams(
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.05,
    max_tokens=4096
)
print('✓ Google Gemma 3 27B 旗舰模型加载就绪！')


In [ ]:
# ==========================================
# 3. 针对 Gemma 3 优化的翻译与 JSON 约束引擎
# ==========================================
import json
import re

GLOSSARY = {
    'Senate': '参议院',
    'House of Representatives': '众议院',
    'majority leader': '多数党领袖',
    'minority leader': '少数党领袖',
    'whip': '党鞭',
    'seniority system': '资历制度',
    'filibuster': '阻挠议事',
    'Federal Power Commission': '联邦电力委员会',
    'Preparedness Investigating Subcommittee': '战备调查特别小组委员会',
    'New Deal': '新政',
    'Southern Caucus': '南方党团',
    'Richard Russell': '理查德·罗素',
    'Leland Olds': '利兰·奥尔兹',
    'Lyndon Baines Johnson': '林登·贝恩斯·约翰逊'
}

SYSTEM_PROMPT = f'''You are a master literary translator specializing in 20th-century American political biography and Robert A. Caro's writing style.
Task: Translate the provided JSON array of English paragraphs into publication-grade Simplified Chinese.

STRICT GUIDELINES:
1. Be completely faithful to the original text, preserving the author's rhetorical rhythm, long-sentence logic, and em-dash pacing;
2. Avoid awkward translationese; use dignified, authentic Chinese publishing prose;
3. Preserve all inline HTML tags (like <em>, <b>, <a>) verbatim;
4. Enforce this terminology mapping: {json.dumps(GLOSSARY, ensure_ascii=False)};
5. OUTPUT FORMAT MUST BE STRICTLY A VALID JSON ARRAY OF STRINGS: ["paragraph 1", "paragraph 2", ...].
The number of elements in the output JSON array must match the input array EXACTLY (1:1).
Do NOT include any commentary, explanations, or notes outside the JSON array.
'''

def extract_json_array(text):
    '''鲁棒提取 JSON 数组'''
    text = text.strip()
    match = re.search(r'```(?:json)?\s*(\[[\s\S]*?\])\s*```', text)
    if match:
        text = match.group(1)
    else:
        start = text.find('[')
        end = text.rfind(']')
        if start != -1 and end != -1 and end > start:
            text = text[start:end+1]
    try:
        data = json.loads(text)
        if isinstance(data, list):
            return [str(item) for item in data]
    except Exception:
        try:
            fixed = re.sub(r',\s*\]', ']', text)
            data = json.loads(fixed)
            if isinstance(data, list):
                return [str(item) for item in data]
        except Exception:
            pass
    return None

def count_chinese_chars(text_list):
    combined = ''.join(str(t) for t in text_list)
    return len(re.findall(r'[一-龥]', combined))

def translate_batch_gemma(blocks, context_prev='', context_next='', max_retries=2):
    '''
    使用 Google Gemma 3 原生 Chat Template 推理翻译
    '''
    user_content = f'''Context before: {context_prev or 'None'}
Context after: {context_next or 'None'}

Input JSON array to translate ({len(blocks)} items):
{json.dumps(blocks, ensure_ascii=False)}

Return ONLY the JSON array of translated Chinese strings:'''

    prompt = f'<start_of_turn>user\n{SYSTEM_PROMPT}\n\n{user_content}<end_of_turn>\n<start_of_turn>model\n'

    for attempt in range(max_retries + 1):
        outputs = llm.generate([prompt], sampling_params)
        raw_output = outputs[0].outputs[0].text.strip()
        parsed = extract_json_array(raw_output)

        if parsed:
            zh_count = count_chinese_chars(parsed)
            if zh_count > 10:
                if len(parsed) == len(blocks):
                    return parsed
                elif len(parsed) < len(blocks):
                    print(f'    ⚠️ 译文段数 ({len(parsed)}) 少于原文 ({len(blocks)})，自动补齐对齐')
                    parsed.extend(['[未对齐段落]'] * (len(blocks) - len(parsed)))
                    return parsed
                else:
                    return parsed[:len(blocks)]
        print(f'    ⚠️ 批次翻译格式不规范（尝试 {attempt+1}/{max_retries+1}），正在重试...')

    # 终极兜底：逐段单独跑，杜绝回退到英文
    print('    ⚠️ 启动单段精准兜底翻译...')
    fallback_res = []
    for single_b in blocks:
        single_p = f'<start_of_turn>user\n{SYSTEM_PROMPT}\n\nTranslate the following paragraph into Chinese directly:\n{single_b}<end_of_turn>\n<start_of_turn>model\n'
        out = llm.generate([single_p], sampling_params)[0].outputs[0].text.strip()
        out = re.sub(r'^(译文[：:]|翻译[：:])\s*', '', out)
        fallback_res.append(out)
    return fallback_res


In [ ]:
# ==========================================
# 4. 全书主控调度引擎（每次均清除缓存并全新重译）
# ==========================================
import os
import glob
import json
import shutil

EPUB_PATH = '/content/drive/MyDrive/Colab Notebooks/translate/RAW/Master of the Senate The Years of Lyndon Johnson, Vol.3.epub'
WORK_DIR = '/content/drive/MyDrive/Colab Notebooks/translate'
FINAL_EPUB = '/content/drive/MyDrive/Colab Notebooks/translate/OUTPUT/Master of the Senate The Years of Lyndon Johnson, Vol.3.epub'

# 核心指令：每次执行均彻底清除历史缓存，从原书重新全新输出！
CLEAR_CACHE_AND_RETRANSLATE = True

if CLEAR_CACHE_AND_RETRANSLATE:
    print('🧹 正在彻底清理历史 cache 目录与 raw 目录，开启全新翻译流程...')
    if os.path.exists(f'{WORK_DIR}/cache'):
        shutil.rmtree(f'{WORK_DIR}/cache')
    if os.path.exists(f'{WORK_DIR}/raw'):
        shutil.rmtree(f'{WORK_DIR}/raw')
    os.makedirs(f'{WORK_DIR}/cache', exist_ok=True)
    os.makedirs(os.path.dirname(FINAL_EPUB), exist_ok=True)
    
    print('📦 从原书重新解包 EPUB 到工作目录...')
    !python3 scripts/epub_tool.py unpack "{EPUB_PATH}" "{WORK_DIR}"

book_info_path = f'{WORK_DIR}/book_info.json'
with open(book_info_path, 'r', encoding='utf-8') as f:
    book_info = json.load(f)

spine = book_info.get('spine', [])
print(f'\n✓ 全新解包完成！全书共有 {len(spine)} 个章节/结构文件，准备开始使用 Gemma 3 27B 全文重译！\n')

# 逐章全新翻译并强制回填
for ch_idx, ch in enumerate(spine):
    xhtml_path = os.path.join(WORK_DIR, 'raw', ch['href'])
    ch_id = f'c_{ch_idx:03d}'
    blocks_file = f'{WORK_DIR}/cache/{ch_id}_blocks.json'
    batches_dir = f'{WORK_DIR}/cache/{ch_id}_batches'
    trans_file = f'{WORK_DIR}/cache/{ch_id}_trans.json'

    print(f'\n🚀 正在全新翻译 [{ch_idx+1}/{len(spine)}] 章节：{ch["href"]} ...')
    
    # 抽取段落与按 1000 词切片
    !python3 scripts/epub_tool.py extract-blocks "{xhtml_path}" -o "{blocks_file}"
    !python3 scripts/epub_tool.py slice-batches "{blocks_file}" "{batches_dir}" --max-words 1000

    manifest_file = f'{batches_dir}/batches_manifest.json'
    if not os.path.exists(manifest_file):
        continue

    with open(manifest_file, 'r', encoding='utf-8') as f:
        manifest = json.load(f)

    chapter_trans = []
    for b_idx, b_info in enumerate(manifest):
        batch_path = os.path.join(batches_dir, b_info['file'])
        with open(batch_path, 'r', encoding='utf-8') as f:
            b_data = json.load(f)

        res = translate_batch_gemma(
            b_data['blocks'],
            context_prev=b_data.get('context_prev', ''),
            context_next=b_data.get('context_next', '')
        )
        chapter_trans.extend(res)
        print(f'  - 批次 {b_idx+1}/{len(manifest)} 翻译完成（累计 {len(chapter_trans)} 段）')

    # 保存全新译文 JSON
    with open(trans_file, 'w', encoding='utf-8') as f:
        json.dump(chapter_trans, f, ensure_ascii=False, indent=2)

    # 100% 强制注入 XHTML 正文，应用出版级 2em 首行缩进
    !python3 scripts/epub_tool.py inject-blocks "{xhtml_path}" "{trans_file}" "{xhtml_path}" --mode mono --indent 2em
    print(f'✓ 章节 {ch_id} 排版回填注入成功！')

# 全局样式修复与重打包
print('\n' + '='*50)
print('正在执行全局出版级样式注入（GB/T 15834 首行缩进 2em）...')
!python3 scripts/epub_tool.py fix-style "{WORK_DIR}" --mode mono --indent 2em

print('正在规范重打包为国际标准 EPUB...')
!python3 scripts/epub_tool.py pack "{WORK_DIR}" "{FINAL_EPUB}"

print('='*50)
print('🎉 全书全新重译、排版与打包全部完成！成品路径：')
print(FINAL_EPUB)
print('='*50)


In [ ]:
# ==========================================
# 5. 全书质量与字数终审巡检报表
# ==========================================
import zipfile
import re
import os

FINAL_EPUB = '/content/drive/MyDrive/Colab Notebooks/translate/OUTPUT/Master of the Senate The Years of Lyndon Johnson, Vol.3.epub'

if not os.path.exists(FINAL_EPUB):
    print(f'❌ 未找到成品文件: {FINAL_EPUB}')
else:
    print(f'🔍 正在对最新生成的 EPUB 进行全盘深度质检与字数审计:\n')
    total_zh_chars = 0
    valid_chapters = 0
    try:
        with zipfile.ZipFile(FINAL_EPUB, 'r') as z:
            print(f'{"章节文件名":<50} | {"中文字数":<10} | {"状态"}')
            print('-' * 75)
            for name in sorted(z.namelist()):
                if name.endswith(('.htm', '.html', '.xhtml')):
                    with z.open(name) as f:
                        content = f.read().decode('utf-8', errors='ignore')
                    count = len(re.findall(r'[一-龥]', content))
                    total_zh_chars += count
                    if count >= 100:
                        status = '✓ 翻译饱满'
                        valid_chapters += 1
                    elif count > 0:
                        status = 'ℹ 短篇/题记'
                    else:
                        status = '⚠️ 占位页/封面'
                    print(f'{name:<50} | {count:<10} 字 | {status}')
        
        print('\n' + '='*50)
        print(f'📊 质量审计汇总报告：')
        print(f'  • 全书总汉字数：{total_zh_chars:,} 字')
        print(f'  • 有效正文章节：{valid_chapters} 篇')
        print(f'  • 排版规范：GB/T 15834 出版级 2em 首行缩进、深浅模式自适应、全 DOM 原位保留')
        print('='*50)
    except Exception as e:
        print(f'❌ 扫描出错: {str(e)}')
